# Przygotowanie danych dla zbioru Mushroom

### Preprocessing danych i analiza wstepna:
1. [Import bibliotek](#0)
2. [Wczytanie danych](#1)
3. [Podstawowe informacje o zbiorze](#2)
4. [Analiza zbalansowania klas](#3)
5. [Podzial danych](#4)
6. [Analiza brakujacych wartosci](#5)
7. [Preprocessing danych](#6)
8. [Wizualizacja wybranych cech](#7)
9. [Korelacja cech modelu](#8)
10. [Podglad wynikow wszystkich modeli](#9)


### <a name='0'></a> Import bibliotek

W pierwszym kroku importujemy biblioteki potrzebne do:
- pracy z danymi tabelarycznymi,
- przygotowania danych do modelu,
- trenowania klasyfikatorow,
- oceny skutecznosci modeli.

Wspolna logika zostala wyniesiona do pliku `mushroom_analysis.py`,
aby wszystkie notebooki korzystaly z tej samej procedury preprocessingu.


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from mushroom_analysis import (
    TARGET_COLUMN,
    BernoulliNB,
    DecisionTreeClassifier,
    KNeighborsClassifier,
    LogisticRegression,
    RANDOM_STATE,
    RandomForestClassifier,
    class_balance_table,
    load_data,
    load_dataset,
    missing_values_table,
    oversample_minority_class,
    plot_class_distribution,
    plot_confusion_matrix_for_model,
    plot_feature_histograms,
    plot_top_feature_correlations,
    preprocess_after_split,
    split_dataset,
)


### <a name='1'></a> Wczytanie danych

Zbior `agaricus-lepiota.data` zawiera informacje o cechach grzybow
oraz zmiennej docelowej `class`, ktora informuje, czy dany grzyb
jest jadalny (`e`) czy trujacy (`p`).


In [ ]:
df = load_data()
df.head()


### <a name='2'></a> Podstawowe informacje o zbiorze

Na tym etapie sprawdzamy rozmiar zbioru. Pozwala to potwierdzic,
ile obserwacji i ile cech bedzie bralo udzial w analizie.


In [ ]:
df.shape


### <a name='3'></a> Analiza zbalansowania klas

Przed budowa modelu nalezy sprawdzic, czy klasy w zbiorze sa rownomiernie
reprezentowane. Silna nierownowaga moglaby prowadzic do zawyzonych metryk
i wymagalaby dodatkowych technik balansowania.


In [ ]:
class_balance_table(df[TARGET_COLUMN])


In [ ]:
plot_class_distribution(df[TARGET_COLUMN])


In [ ]:
balance_df = class_balance_table(df[TARGET_COLUMN])
edible_row = balance_df[balance_df["class_code"] == "e"].iloc[0]
poisonous_row = balance_df[balance_df["class_code"] == "p"].iloc[0]

print(
    f"Wnioski: w zbiorze znajduje sie {int(edible_row['count'])} grzybow jadalnych "
    f"({edible_row['percentage']:.1f}%) oraz {int(poisonous_row['count'])} grzybow trujacych "
    f"({poisonous_row['percentage']:.1f}%). Oznacza to, ze klasy sa prawie zbalansowane."
)


### <a name='4'></a> Podzial danych na zbiory treningowy, walidacyjny i testowy

Zgodnie z zalozeniami projektowymi dzielimy dane w proporcji `70/15/15`.
Podzial wykonywany jest przed imputacja brakow, aby uniknac przecieku danych.


In [ ]:
train_df, validation_df, test_df = split_dataset(df)

pd.DataFrame(
    [
        {"split": "train", "rows": len(train_df)},
        {"split": "validation", "rows": len(validation_df)},
        {"split": "test", "rows": len(test_df)},
    ]
)


### <a name='5'></a> Analiza brakujacych wartosci

W zbiorze mushroom brakujace wartosci sa szczegolnie istotne,
poniewaz niektore cechy kategorii zawieraja symbole `?`.
Po podziale danych sprawdzamy, gdzie wystepuja braki i w jakiej skali.


In [ ]:
missing_values_table(
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df),
)


In [ ]:
missing_df = missing_values_table(
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df),
)

total_missing = int(missing_df["missing_count"].sum())
train_missing = int(missing_df.loc[missing_df["split"] == "train", "missing_count"].sum())
validation_missing = int(missing_df.loc[missing_df["split"] == "validation", "missing_count"].sum())
test_missing = int(missing_df.loc[missing_df["split"] == "test", "missing_count"].sum())

print(
    f"Wnioski: wykryto lacznie {total_missing} brakujacych wartosci, "
    f"w tym {train_missing} w zbiorze treningowym, {validation_missing} w walidacyjnym "
    f"oraz {test_missing} w testowym. Braki wystepuja w kolumnie 'stalk_root', "
    f"dlatego imputacja po podziale danych jest konieczna."
)


### <a name='6'></a> Preprocessing danych

W tej sekcji wykonujemy:
- imputacje najczesciej wystepujacej kategorii,
- kodowanie zmiennej docelowej do postaci numerycznej,
- kodowanie cech kategorycznych z wykorzystaniem One-Hot Encoding.

Taki sposob przygotowania danych jest odpowiedni dla analizowanych modeli.


In [ ]:
preprocessed = preprocess_after_split(train_df, validation_df, test_df)

print("X_train_encoded:", preprocessed["X_train_encoded"].shape)
print("X_validation_encoded:", preprocessed["X_validation_encoded"].shape)
print("X_test_encoded:", preprocessed["X_test_encoded"].shape)


### <a name='7'></a> Wizualizacja wybranych cech

Histogramy pomagaja zrozumiec rozklad kategorii dla cech,
ktore moga miec duze znaczenie przy rozroznianiu grzybow jadalnych
i trujacych.


In [ ]:
plot_feature_histograms(train_df)


### <a name='8'></a> Korelacja cech modelu

Po zakodowaniu cech mozemy sprawdzic, ktore z nich wykazuja
najsilniejszy zwiazek ze zmienna docelowa. Pozwala to lepiej
uzasadnic, dlaczego model osiagnie okreslone wyniki.


In [ ]:
top_correlations = plot_top_feature_correlations(
    preprocessed["X_train_encoded"],
    preprocessed["y_train"],
    top_n=12,
)
top_correlations


In [ ]:
strongest = top_correlations.iloc[0]
print(
    f"Wnioski: najsilniej z targetem powiazana jest cecha "
    f"'{strongest['encoded_feature']}' o bezwzglednej korelacji "
    f"{strongest['absolute_target_correlation']:.4f}. Potwierdza to, ze czesc cech "
    f"bardzo dobrze rozroznia klasy jadalne i trujace."
)


### <a name='9'></a> Podglad wynikow wszystkich modeli

Zanim przejdziemy do osobnych notebookow modelowych,
warto zobaczyc zbiorcze porownanie wszystkich pieciu algorytmow.


In [ ]:
results_df = evaluate_models(preprocessed).drop(columns="fitted_model")
results_df


In [ ]:
best_row = results_df.iloc[0]
print(
    f"Wnioski ogolne: najlepszy wynik walidacyjny uzyskal model "
    f"{best_row['model']} z accuracy = {best_row['validation_accuracy']:.4f} "
    f"oraz F1 = {best_row['validation_f1']:.4f}. Oznacza to, ze problem klasyfikacji "
    f"na zbiorze mushroom jest bardzo dobrze separowalny."
)
